# `xy.pyplot` — Matplotlib shim validation notebook

This notebook is an executable acceptance test for the Matplotlib-flavored shim. It checks the complete Matplotlib 3.11 2-D plotting-method inventory, exercises every major chart family, validates compatibility return objects, and confirms that heavy computation is served by the native Rust backend.

Run it from an editable checkout after `uv pip install -e .`. Every section contains assertions; a cell that finishes without an exception has passed.

In [ ]:
import io
import sys

import numpy as np

import xy.pyplot as plt
from xy import kernels

rng = np.random.default_rng(2026)
plt.close("all")

assert kernels.BACKEND == "native"
assert "matplotlib" not in sys.modules, "the shim must not import real Matplotlib"
print("native backend:", kernels.BACKEND)
print("real matplotlib imported:", "matplotlib" in sys.modules)

## 1. Official Matplotlib 3.11 2-D plotting inventory

Both `Axes` and the stateful `pyplot` namespace must expose every method in the upstream `Axes` **Plotting** section.

In [ ]:
PLOTTING_METHODS = [
    "plot",
    "errorbar",
    "scatter",
    "step",
    "loglog",
    "semilogx",
    "semilogy",
    "fill_between",
    "fill_betweenx",
    "bar",
    "barh",
    "bar_label",
    "grouped_bar",
    "stem",
    "eventplot",
    "pie",
    "pie_label",
    "stackplot",
    "broken_barh",
    "vlines",
    "hlines",
    "fill",
    "axhline",
    "axhspan",
    "axvline",
    "axvspan",
    "axline",
    "acorr",
    "angle_spectrum",
    "cohere",
    "csd",
    "magnitude_spectrum",
    "phase_spectrum",
    "psd",
    "specgram",
    "xcorr",
    "ecdf",
    "boxplot",
    "violinplot",
    "bxp",
    "violin",
    "hexbin",
    "hist",
    "hist2d",
    "stairs",
    "clabel",
    "contour",
    "contourf",
    "imshow",
    "matshow",
    "pcolor",
    "pcolorfast",
    "pcolormesh",
    "spy",
    "tripcolor",
    "triplot",
    "tricontour",
    "tricontourf",
    "annotate",
    "text",
    "table",
    "arrow",
    "barbs",
    "quiver",
    "quiverkey",
    "streamplot",
]

missing_axes = [name for name in PLOTTING_METHODS if not hasattr(plt.Axes, name)]
missing_pyplot = [name for name in PLOTTING_METHODS if not hasattr(plt, name)]
assert missing_axes == []
assert missing_pyplot == []
print(f"{len(PLOTTING_METHODS)} / {len(PLOTTING_METHODS)} plotting methods present")

## 2. Basic plots, grouped bars, labels, fills, and spans

In [ ]:
plt.close("all")
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
x = np.linspace(0, 2 * np.pi, 200)

lines = axes[0, 0].plot(x, np.sin(x), "r--o", label="sin(x)")
axes[0, 0].fill_between(x, np.sin(x) - 0.15, np.sin(x) + 0.15, alpha=0.2)
axes[0, 0].axhline(0, color="black", linewidth=0.8)
axes[0, 0].axvspan(np.pi / 2, np.pi, color="gold", alpha=0.15)
axes[0, 0].set_title("line + band + spans")
axes[0, 0].legend()
assert len(lines) == 1

scatter = axes[0, 1].scatter(
    rng.normal(size=300),
    rng.normal(size=300),
    c=np.linspace(0, 1, 300),
    s=np.linspace(12, 100, 300),
    cmap="plasma",
    alpha=0.7,
)
scatter.set_label("samples")
axes[0, 1].set_title("encoded scatter")

grouped = axes[1, 0].grouped_bar(
    [[4, 7, 5], [6, 5, 8], [5, 6, 7]],
    tick_labels=["A", "B", "C"],
    labels=["alpha", "beta", "gamma"],
)
for container in grouped.bar_containers:
    labels = axes[1, 0].bar_label(container, fmt="%.0f")
    assert len(labels) == 3
axes[1, 0].legend()
axes[1, 0].set_title("Matplotlib 3.11 grouped_bar")

axes[1, 1].stackplot(x, np.sin(x) + 1.2, np.cos(x) + 1.2, baseline="wiggle")
axes[1, 1].step(x[::10], np.sin(x[::10]), where="mid", color="black")
axes[1, 1].set_title("native stack layout + step")

fig.suptitle("Basic Matplotlib shim families")
fig.tight_layout()
fig

## 3. Statistical, binned, and precomputed distributions

In [ ]:
plt.close("all")
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
samples = [rng.normal(loc, 0.8, 2_000) for loc in (-1, 0.5, 2)]

box = axes[0, 0].boxplot(samples, showfliers=True)
assert set(box) == {"whiskers", "caps", "boxes", "medians", "fliers", "means"}
axes[0, 0].set_title("boxplot")

violin = axes[0, 1].violinplot(samples, showmeans=True, showmedians=True)
assert "bodies" in violin
axes[0, 1].set_title("violinplot")

values = rng.normal(size=20_000)
weights = rng.uniform(0.1, 2.0, len(values))
axes[0, 2].ecdf(values, weights=weights, complementary=True, color="tab:purple")
axes[0, 2].set_title("native weighted complementary ECDF")

axes[1, 0].hist(values, bins=40, density=True, alpha=0.75)
axes[1, 0].set_title("histogram")

hx = rng.normal(size=50_000)
hy = 0.65 * hx + rng.normal(scale=0.6, size=len(hx))
h, xedges, yedges, image = axes[1, 1].hist2d(hx, hy, bins=(48, 40), cmap="inferno")
assert h.shape == (48, 40) and image is not None
axes[1, 1].set_title("native hist2d")

axes[1, 2].hexbin(hx, hy, gridsize=42, bins="log", cmap="viridis")
axes[1, 2].set_title("native hexbin")

fig.tight_layout()
fig

### Precomputed `bxp` and `violin` return structures

In [ ]:
plt.close("all")
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
stats = [
    {"med": 2, "q1": 1, "q3": 3, "whislo": 0, "whishi": 4, "fliers": [5], "mean": 2.1},
    {"med": 4, "q1": 3, "q3": 5, "whislo": 2, "whishi": 6, "fliers": [], "mean": 4.2},
]
bxp_result = axes[0].bxp(stats, showmeans=True)
assert set(bxp_result) == {"boxes", "medians", "whiskers", "caps", "means", "fliers"}
axes[0].set_title("precomputed bxp")

coords = np.linspace(-2.5, 2.5, 80)
vpstats = [
    {
        "coords": coords,
        "vals": np.exp(-(coords**2)),
        "mean": 0.0,
        "median": 0.0,
        "min": -2.5,
        "max": 2.5,
        "quantiles": [-0.75, 0.75],
    }
]
violin_result = axes[1].violin(vpstats, showmeans=True, showmedians=True)
assert "bodies" in violin_result and "cquantiles" in violin_result
axes[1].set_title("precomputed violin")
fig.tight_layout()
fig

## 4. Images, rectilinear/curvilinear contours, and unstructured triangles

In [ ]:
plt.close("all")
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
gx = np.linspace(-2, 2, 60)
gy = np.linspace(-1.5, 1.5, 48)
xx, yy = np.meshgrid(gx, gy)
z = np.sin(xx * 2) * np.cos(yy * 3)

axes[0, 0].imshow(z, origin="lower", cmap="coolwarm", extent=(-2, 2, -1.5, 1.5))
axes[0, 0].set_title("imshow")

mesh = axes[0, 1].pcolormesh(gx**3, gy**3, z, shading="auto", cmap="viridis")
assert mesh is not None
axes[0, 1].set_title("nonuniform pcolormesh")

contours = axes[0, 2].contour(gx, gy, z, levels=8, cmap="plasma")
labels = axes[0, 2].clabel(contours, fmt="%.2f")
assert labels
axes[0, 2].set_title("contour + clabel")

warped_x = xx + 0.18 * np.sin(yy * 2)
warped_y = yy + 0.12 * np.cos(xx * 2)
axes[1, 0].contourf(warped_x, warped_y, z, levels=10, cmap="inferno")
axes[1, 0].set_title("curvilinear contourf → native topology")

tx = rng.uniform(-1, 1, 120)
ty = rng.uniform(-1, 1, 120)
tz = np.sin(tx * 4) + np.cos(ty * 3)
axes[1, 1].tripcolor(tx, ty, tz, cmap="turbo")
axes[1, 1].triplot(tx, ty, color="white", linewidth=0.25)
axes[1, 1].set_title("tripcolor + native Delaunay")

tri_contour = axes[1, 2].tricontour(tx, ty, tz, levels=7, cmap="coolwarm")
assert len(tri_contour.levels) == 7
axes[1, 2].set_title("tricontour")
fig.tight_layout()
fig

## 5. Pie containers, pie labels, and tables

In [ ]:
plt.close("all")
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
pie = axes[0].pie(
    [2, 3, 5],
    labels=["alpha", "beta", "gamma"],
    wedgeprops={"width": 0.38, "edgecolor": "white", "linewidth": 0.8},
    startangle=90,
)
extra_labels = axes[0].pie_label(pie, "{absval:g} ({frac:.0%})", distance=0.72)
assert len(pie.wedges) == 3
np.testing.assert_allclose(pie.fracs, [0.2, 0.3, 0.5])
assert len(extra_labels) == 3 and len(pie.texts) == 2
axes[0].set_title("Matplotlib 3.11 PieContainer")

table = axes[1].table(
    cellText=[["North", 12], ["South", 18], ["West", 15]],
    colLabels=["Region", "Value"],
    cellColours=[
        ["#dbeafe", "#dbeafe"],
        ["#dcfce7", "#dcfce7"],
        ["#fef3c7", "#fef3c7"],
    ],
)
assert len(table.get_celld()) == 8
axes[1].set_title("table via generic mesh + text")
fig.tight_layout()
fig

## 6. Vector fields and native streamline integration

In [ ]:
plt.close("all")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
vx = np.linspace(-2, 2, 14)
vy = np.linspace(-2, 2, 11)
vxx, vyy = np.meshgrid(vx, vy)
u, v = -vyy, vxx
speed = np.hypot(u, v)

quiver = axes[0].quiver(vx, vy, u, v, speed, cmap="plasma", angles="xy", scale_units="xy")
key = axes[0].quiverkey(quiver, 0.5, 0.5, 1.0, "1 unit")
assert key is not None
axes[0].set_title("quiver + quiverkey")

axes[1].barbs(vx, vy, u, v, color="tab:blue")
axes[1].set_title("barbs")

stream = axes[2].streamplot(vx, vy, u, v, density=0.8, maxlength=3.0, color="tab:green")
assert stream.lines is stream.arrows
axes[2].set_title("native streamplot")
fig.tight_layout()
fig

## 7. Native spectral and correlation family

In [ ]:
plt.close("all")
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
sample_rate = 1024.0
time = np.arange(4096) / sample_rate
signal = np.sin(2 * np.pi * 64 * time) + 0.25 * np.sin(2 * np.pi * 180 * time)
shifted = np.sin(2 * np.pi * 64 * time + 0.4)

magnitude, frequency, _ = axes[0, 0].magnitude_spectrum(signal, Fs=sample_rate, pad_to=512)
assert frequency[np.argmax(magnitude)] == 64.0
axes[0, 0].set_title("magnitude_spectrum")

phase, phase_frequency, _ = axes[0, 1].phase_spectrum(signal, Fs=sample_rate, pad_to=512)
assert len(phase) == len(phase_frequency)
axes[0, 1].set_title("phase_spectrum")

power, power_frequency = axes[0, 2].psd(signal, NFFT=256, Fs=sample_rate, noverlap=128)
assert power_frequency[np.argmax(power)] == 64.0
axes[0, 2].set_title("psd")

coherence, coherence_frequency = axes[1, 0].cohere(
    signal, shifted, NFFT=256, Fs=sample_rate, noverlap=128
)
assert coherence[np.flatnonzero(coherence_frequency == 64.0)[0]] > 0.99
axes[1, 0].set_title("cohere")

spectrum, frequencies, times, image = axes[1, 1].specgram(
    signal, NFFT=256, Fs=sample_rate, noverlap=128, cmap="inferno"
)
assert spectrum.shape == (129, 31) and image is not None
axes[1, 1].set_title("specgram")

lags, corr, lines, baseline = axes[1, 2].xcorr(signal, shifted, maxlags=32)
assert len(lags) == len(corr) == 65 and lines is not None and baseline is not None
axes[1, 2].set_title("xcorr")
fig.tight_layout()
fig

## 8. Export and artist-mutation smoke tests

The shim should produce browser-free PNG, SVG, and standalone HTML while preserving Matplotlib-style mutable handles.

In [ ]:
plt.close("all")
fig, ax = plt.subplots(figsize=(6, 3.5))
(line,) = ax.plot([0, 1, 2, 3], [1, 3, 2, 4], "o-", label="before")
line.set_ydata([2, 1, 4, 3])
line.set_color("tab:orange")
line.set_label("after")
ax.legend()

png = io.BytesIO()
svg = io.BytesIO()
fig.savefig(png, format="png")
fig.savefig(svg, format="svg")
html = fig._to_html()

assert png.getvalue().startswith(b"\x89PNG")
assert svg.getvalue().lstrip().startswith(b"<svg")
assert "<!doctype html>" in html.lower()
print("PNG bytes:", len(png.getvalue()))
print("SVG bytes:", len(svg.getvalue()))
print("HTML bytes:", len(html.encode()))
fig

## Result

If every cell above passed, the notebook verified:

- complete Matplotlib 3.11 2-D plotting-method presence on `Axes` and `pyplot`;
- representative rendering across basic, statistical, binned, mesh, triangle, pie, table, vector, and spectral families;
- Matplotlib 3.11 compatibility containers and mutable artist handles;
- native Rust backend dispatch without importing real Matplotlib; and
- PNG, SVG, and standalone HTML export.